# 09 · Reentrenamiento periódico de LightGBM

En `04` y `06`, LightGBM se entrena una sola vez (13-jul-2015) y pronostica 9 meses sin actualizarse. Aquí se **reentrena cada 4 semanas** con toda la historia disponible hasta ese momento; cada revisión usa el modelo más reciente. Los rezagos son de 28 días o más, así que ningún pronóstico usa información posterior al origen.

Los demás pronósticos se leen de `06_ablacion_censura` (`data/processed/pronosticos_13d_CA_1.parquet`).

In [1]:
from pathlib import Path
from datetime import timedelta
import sys
import time
import numpy as np
import pandas as pd
import polars as pl
import lightgbm as lgb
import plotly.express as px

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
PROC = ROOT / "data" / "processed"
FIG_DIR = ROOT / "reports" / "figures"
REP = ROOT / "reports"
if str(ROOT / "src") not in sys.path:
    sys.path.append(str(ROOT / "src"))
from features import construir_features, FEATS, CAT_FEATS
from simulate import simular_rs

# --- Supuestos de la política (R, S) ---
R, L = 7, 7               # revisión semanal, lead time de 7 días
PROT = R + L - 1          # días que cubre cada pedido: del día siguiente hasta antes del próximo arribo
N_CALIB, N_SIM = 13, 26   # semanas de calibración y de simulación
H_ANUAL = 0.25            # costo de mantener: 25% anual del precio
MARGEN = 0.30             # costo de faltante: margen perdido por unidad no vendida
WARMUP = 14               # días iniciales que no se miden
ALPHAS = [0.50, 0.70, 0.80, 0.90, 0.95, 0.98, 0.99]

df = pl.read_parquet(PROC / "m5_CA_1.parquet")
seg = pl.read_parquet(PROC / "segmentacion_CA_1.parquet")
mascara = pl.read_parquet(PROC / "mascara_CA_1.parquet")
ids_validos = (pd.read_parquet(PROC / "cv_baselines_CA_1.parquet", columns=["unique_id"])
                 ["unique_id"].unique().tolist())

# --- Calendario (todo antes de la prueba final reservada) ---
dev_end = pd.Timestamp(df["date"].max()) - timedelta(days=28)
last_review = dev_end - timedelta(days=PROT)
reviews_sim = [last_review - timedelta(days=7 * k) for k in range(N_SIM)][::-1]
sim_start = reviews_sim[0]
last_calib = sim_start - timedelta(days=PROT + 1)
origins_calib = [last_calib - timedelta(days=7 * k) for k in range(N_CALIB)][::-1]
train_end = origins_calib[0]
origenes = pd.date_range(origins_calib[0], last_review, freq="7D")
N_WINDOWS = len(origenes)

print("Entrenamiento LightGBM hasta:", train_end.date())
print("Calibración (orígenes):      ", origins_calib[0].date(), "→", last_calib.date(), f"({N_CALIB})")
print("Simulación (revisiones):     ", sim_start.date(), "→", last_review.date(), f"({N_SIM})")
print("Último día simulado:         ", dev_end.date(), f"| orígenes totales: {N_WINDOWS}")

Entrenamiento LightGBM hasta: 2015-07-13
Calibración (orígenes):       2015-07-13 → 2015-10-05 (13)
Simulación (revisiones):      2015-10-19 → 2016-04-11 (26)
Último día simulado:          2016-04-24 | orígenes totales: 40


In [2]:
dev = (df.filter(pl.col("date") <= dev_end.date())
         .with_columns(pl.col("id").cast(pl.Utf8))
         .filter(pl.col("id").is_in(ids_validos)))

inicio = dev.group_by("id").agg(inicio=pl.col("date").min())
fuera = (mascara.filter((pl.col("date") > train_end.date())
                        & pl.col("tipo").is_in(["inactivo", "pre_lanzamiento"]))
                ["id"].unique().to_list())
universo = sorted(
    inicio.filter(pl.col("inicio") <= (train_end - timedelta(days=112)).date())
          .filter(~pl.col("id").is_in(fuera))["id"].to_list()
)
print(f"Productos simulados: {len(universo):,} de {len(ids_validos):,} "
      f"(excluidos por inactividad en el periodo: {len(fuera):,})")

d = (
    dev.filter(pl.col("id").is_in(universo))
    .join(mascara.select(["id", "date", "tipo", "excluir"]), on=["id", "date"], how="left")
    .with_columns(pl.col("excluir").fill_null(False), pl.col("tipo").fill_null("normal"),
                  ventas=pl.col("sales").cast(pl.Float64))
    .sort(["id", "date"])
    .with_columns(limpia=pl.when(pl.col("excluir")).then(pl.lit(None)).otherwise(pl.col("ventas")))
    .with_columns(
        num=pl.col("limpia").fill_null(0).rolling_sum(28).shift(1).over("id"),
        den=(~pl.col("excluir")).cast(pl.Float64).rolling_sum(28).shift(1).over("id"),
        media_prod=pl.col("limpia").mean().over("id"),
    )
    .with_columns(imp=pl.when(pl.col("den") > 0).then(pl.col("num") / pl.col("den"))
                        .otherwise(pl.col("media_prod")))
    .with_columns(demanda=pl.when(pl.col("tipo") == "agotado").then(pl.col("imp"))
                            .otherwise(pl.col("ventas")))
    # Navidad: la tienda cierra, la demanda real es 0
    .with_columns(demanda=pl.when((pl.col("date").dt.month() == 12) & (pl.col("date").dt.day() == 25))
                            .then(0.0).otherwise(pl.col("demanda")))
)

ventana = d.filter(pl.col("date") > train_end.date())
print(f"Días-producto con demanda imputada (agotado) en calibración + simulación: "
      f"{(ventana['tipo'] == 'agotado').mean():.1%}")

Productos simulados: 2,467 de 3,041 (excluidos por inactividad en el periodo: 569)


Días-producto con demanda imputada (agotado) en calibración + simulación: 2.6%


In [3]:
PARAMS = dict(objective="tweedie", tweedie_variance_power=1.1,
              n_estimators=500, learning_rate=0.05, num_leaves=127, min_child_samples=100,
              subsample=0.8, subsample_freq=1, colsample_bytree=0.8, reg_lambda=1.0,
              n_jobs=-1, verbose=-1)
CADA = 4   # semanas entre reentrenamientos

pdf = construir_features(dev, mascara)
reentrenos = list(origenes[::CADA])
fechas = pd.date_range(origenes[0] + timedelta(days=1), dev_end)

def suma_por_origen(M, orig):
    """Suma de los días o+1 … o+PROT para cada origen o."""
    return pd.DataFrame({o: M.loc[o + timedelta(days=1): o + timedelta(days=PROT)].sum() for o in orig}).T

bloques = []
t0 = time.time()
for i, r in enumerate(reentrenos):
    tr = pdf[(pdf["date"] <= r) & (~pdf["excluir"]) & (pdf["navidad"] == 0)]
    modelo = lgb.LGBMRegressor(**PARAMS).fit(tr[FEATS], tr["sales"], categorical_feature=CAT_FEATS)
    hasta = reentrenos[i + 1] if i + 1 < len(reentrenos) else origenes[-1] + timedelta(days=1)
    orig = [o for o in origenes if r <= o < hasta]          # revisiones que usan este modelo
    te = pdf[(pdf["date"] > r) & (pdf["date"] <= orig[-1] + timedelta(days=PROT)) & (pdf["id"].isin(universo))].copy()
    te["p"] = np.clip(modelo.predict(te[FEATS]), 0, None)
    P = te.pivot(index="date", columns="id", values="p").reindex(index=fechas, columns=universo)
    bloques.append(suma_por_origen(P, orig))
    print(f"Modelo {i + 1}/{len(reentrenos)}: entrenado hasta {r.date()} con {len(tr):,} filas → "
          f"{len(orig)} revisiones | {(time.time() - t0) / 60:.1f} min")

F_re = pd.concat(bloques).reindex(index=origenes, columns=universo)
assert not F_re.isna().any().any(), "Huecos en los pronósticos reentrenados"

Modelo 1/10: entrenado hasta 2015-07-13 con 3,317,042 filas → 4 revisiones | 1.2 min


Modelo 2/10: entrenado hasta 2015-08-10 con 3,392,402 filas → 4 revisiones | 2.5 min


Modelo 3/10: entrenado hasta 2015-09-07 con 3,467,000 filas → 4 revisiones | 3.6 min


Modelo 4/10: entrenado hasta 2015-10-05 con 3,542,563 filas → 4 revisiones | 4.9 min


Modelo 5/10: entrenado hasta 2015-11-02 con 3,618,147 filas → 4 revisiones | 6.3 min


Modelo 6/10: entrenado hasta 2015-11-30 con 3,693,045 filas → 4 revisiones | 7.7 min


Modelo 7/10: entrenado hasta 2015-12-28 con 3,764,803 filas → 4 revisiones | 8.9 min


Modelo 8/10: entrenado hasta 2016-01-25 con 3,840,365 filas → 4 revisiones | 10.1 min


Modelo 9/10: entrenado hasta 2016-02-22 con 3,916,521 filas → 4 revisiones | 11.3 min


Modelo 10/10: entrenado hasta 2016-03-21 con 3,994,249 filas → 4 revisiones | 12.7 min


In [4]:
# --- Demanda, precios y pronósticos guardados por 06 ---
pron = pd.read_parquet(PROC / "pronosticos_13d_CA_1.parquet")
tabla13 = {m: g.pivot(index="origen", columns="id", values="unidades").reindex(index=origenes, columns=universo)
           for m, g in pron.groupby("modelo")}
A = tabla13.pop("real")
F = {"LGB": tabla13["LGB"], "LGB_reentrenado": F_re,
     "AutoETS_imp": tabla13["AutoETS_imp"], "TSB_imp": tabla13["TSB_imp"], "TSB": tabla13["TSB"]}

pdd = d.select(["id", "date", "demanda", "sell_price"]).to_pandas()
pdd["date"] = pd.to_datetime(pdd["date"]).astype("datetime64[ns]")
D_full = pdd.pivot(index="date", columns="id", values="demanda").reindex(index=fechas, columns=universo)
precio_full = pdd.pivot(index="date", columns="id", values="sell_price").reindex(index=fechas, columns=universo)
assert np.allclose(suma_por_origen(D_full, origenes).to_numpy(), A.to_numpy()), "La demanda no coincide con 06"

calib = [o for o in origenes if o <= last_calib]
simo = [o for o in origenes if o >= sim_start]
for m in ["LGB", "LGB_reentrenado"]:
    e = F[m].loc[simo] - A.loc[simo]
    print(f"{m:16} sesgo en simulación: {e.sum().sum() / A.loc[simo].sum().sum():+.2%} | "
          f"MAE 13 días: {e.abs().mean().mean():.2f} u.")

LGB              sesgo en simulación: +0.39% | MAE 13 días: 6.48 u.
LGB_reentrenado  sesgo en simulación: +0.04% | MAE 13 días: 6.04 u.


In [5]:
clase = (seg.select(id=pl.col("id").cast(pl.Utf8), sb_class="sb_class").to_pandas()
            .set_index("id")["sb_class"].reindex(universo))

q = {}
for m, Fm in F.items():
    z = (A.loc[calib] - Fm.loc[calib]) / np.sqrt(Fm.loc[calib] + 1)
    zl = z.T.rename_axis(index="id", columns="origen").stack().rename("z").reset_index()
    zl["clase"] = zl["id"].map(clase)
    q[m] = zl.groupby("clase")["z"].quantile(ALPHAS).unstack()


sim_fechas = pd.date_range(sim_start, dev_end)
D_sim = D_full.loc[sim_fechas].to_numpy()
precio = precio_full.loc[sim_fechas].to_numpy()
revision = sim_fechas.isin(reviews_sim)
idx_rev = np.where(revision)[0]
assert idx_rev[0] == 0 and len(idx_rev) == N_SIM

def metricas(res, cols=slice(None)):
    s = slice(WARMUP, None)
    dem, vend, perd, inv, p = (x[s][:, cols] for x in
                               (D_sim, res["vendido"], res["perdido"], res["inventario"], precio))
    c_mant = (inv * p * H_ANUAL / 365).sum()
    c_falt = (perd * p * MARGEN).sum()
    return {"fill_rate": vend.sum() / dem.sum(),
            "inv_valor_prom": (inv * p).sum(axis=1).mean(),
            "dias_cobertura": inv.sum() / dem.sum(),
            "costo_mantener": c_mant,
            "costo_faltante": c_falt,
            "costo_total": c_mant + c_falt}


N_BINS = 5

def calibrar_por_volumen(Fm_all):
    """Cuantiles de z por (clase, quintil de volumen) con los errores fuera de muestra de la calibración."""
    Fc = Fm_all.loc[calib]
    z = (A.loc[calib] - Fc) / np.sqrt(Fc + 1)
    zl = pd.DataFrame({"id": np.tile(universo, len(calib)),
                       "f": Fc.to_numpy().ravel(), "z": z.to_numpy().ravel()})
    zl["clase"] = zl["id"].map(clase)
    edges = np.unique(np.quantile(zl["f"], np.linspace(0, 1, N_BINS + 1)))
    edges[0], edges[-1] = -np.inf, np.inf
    zl["bin"] = pd.cut(zl["f"], edges, labels=False).astype(int)
    return zl.groupby(["clase", "bin"])["z"].quantile(ALPHAS).unstack(), edges


filas = []
for m, Fm_all in F.items():
    tabla_q, edges = calibrar_por_volumen(Fm_all)
    Fm = Fm_all.loc[reviews_sim].to_numpy()
    bins = pd.cut(Fm.ravel(), edges, labels=False).astype(int)
    claves = pd.MultiIndex.from_arrays([np.tile(clase.to_numpy(), len(reviews_sim)), bins])
    for a in ALPHAS:
        qa = tabla_q[a].reindex(claves).to_numpy().reshape(Fm.shape)
        qa = np.where(np.isnan(qa), np.tile(clase.map(q[m][a]).to_numpy(), (len(reviews_sim), 1)), qa)
        S = np.zeros_like(D_sim)
        S[idx_rev] = np.maximum(0.0, Fm + qa * np.sqrt(Fm + 1))
        res = simular_rs(D_sim, S, revision, L, inv_inicial=np.ceil(S[0]))
        filas.append({"modelo": m, "alpha": a, **metricas(res)})
tabla = pd.DataFrame(filas)

# Control: los modelos leídos de 06 reproducen su simulación
pub = pd.read_csv(REP / "ablacion_censura_CA_1.csv")
chk = pub.merge(tabla, on=["modelo", "alpha"], suffixes=("_06", "_09"))
assert len(chk) == 4 * len(ALPHAS) and np.allclose(chk["costo_total_06"], chk["costo_total_09"], rtol=1e-6)
print("Control OK: reproduce 06_ablacion_censura")

def inv_en(modelo, fr):
    c = tabla.query("modelo == @modelo").sort_values("fill_rate")
    return np.interp(fr, c["fill_rate"], c["inv_valor_prom"])

opt = tabla.loc[tabla.groupby("modelo")["costo_total"].idxmin()].set_index("modelo")
res_re = pd.DataFrame({"Inv_FR95": {m: inv_en(m, 0.95) for m in F}, "Alpha_opt": opt["alpha"],
                       "Fill_opt": opt["fill_rate"], "Costo_opt": opt["costo_total"]}).loc[list(F)]
res_re["vs_LGB_%_costo"] = (res_re["Costo_opt"] / res_re.loc["LGB", "Costo_opt"] - 1) * 100
res_re["vs_LGB_%_inv95"] = (res_re["Inv_FR95"] / res_re.loc["LGB", "Inv_FR95"] - 1) * 100
tabla.to_csv(REP / "reentrenamiento_CA_1.csv", index=False)
res_re.to_csv(REP / "reentrenamiento_resumen_CA_1.csv")
res_re.round({"Inv_FR95": 0, "Fill_opt": 4, "Costo_opt": 0, "vs_LGB_%_costo": 1, "vs_LGB_%_inv95": 1})

Control OK: reproduce 06_ablacion_censura


,Inv_FR95,Alpha_opt,Fill_opt,Costo_opt,vs_LGB_%_costo,vs_LGB_%_inv95
LGB,120675.0,0.98,0.9900,31977.0,0.0,0.0
LGB_reentrenado,114837.0,0.98,0.9897,30968.0,-3.2,-4.8
AutoETS_imp,121055.0,0.98,0.9893,31169.0,-2.5,0.3
TSB_imp,122834.0,0.98,0.9900,31333.0,-2.0,1.8
TSB,161137.0,0.95,0.9685,43273.0,35.3,33.5


## Conclusiones

| Pronóstico | Inventario a fill rate 95% | Costo óptimo | vs. LightGBM entrenado una vez |
|---|---|---|---|
| LightGBM entrenado una vez | $120,675 | $31,977 | — |
| **LightGBM reentrenado cada 4 semanas** | **$114,837** | **$30,968** | −4.8% inventario, −3.2% costo |
| AutoETS con demanda corregida | $121,055 | $31,169 | |
| TSB con demanda corregida | $122,834 | $31,333 | |
| TSB con ventas | $161,137 | $43,273 | |

- Reentrenar reduce el sesgo en la simulación (+0.39% → +0.04%) y el error absoluto a 13 días (6.48 → 6.04 unidades).
- Con reentrenamiento, LightGBM queda ligeramente por delante de los clásicos corregidos (−0.6% de costo y −5% de inventario frente a AutoETS). Es una comparación más justa, porque TSB y AutoETS ya actualizaban su estado cada semana. Aun así, la diferencia es pequeña frente al efecto de corregir la censura (−24% a −28%) y no se probó su significancia.